# Diabetes Prediction Using Classification Models

## Introduction

The goal of this assignment is to predict whether a person has diabetes using demographic and medical information from the Diabetes Prediction Dataset. Four classification models are trained and compared: Logistic Regression, Decision Tree, Random Forest, and K-Nearest Neighbors.

The models are evaluated using Accuracy, Precision, Recall, F1-score, and ROC-AUC. After comparing the results, the best-performing model is selected and its confusion matrix is displayed.


## 1. Import the Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")


## 2. Load the Dataset

The code below loads the diabetes dataset from the same folder as this notebook. The expected filename is `diabetes_prediction_dataset.csv`.


In [ ]:
possible_files = [
    "diabetes_prediction_dataset.csv",
    "Diabetes_Prediction_Dataset.csv",
    "diabetes.csv"
]

file_path = next(
    (file_name for file_name in possible_files if os.path.exists(file_name)),
    None
)

if file_path is None:
    raise FileNotFoundError(
        "The diabetes CSV file was not found. Place "
        "'diabetes_prediction_dataset.csv' in the same folder as this notebook."
    )

diabetes_data = pd.read_csv(file_path)

print(f"Dataset loaded successfully from: {file_path}")
print(f"The dataset contains {diabetes_data.shape[0]} rows and {diabetes_data.shape[1]} columns.")

diabetes_data.head()


## 3. Review and Prepare the Data

Before training the models, duplicate rows are removed and the target column is separated from the predictor variables. The target column is `diabetes`, where `0` means the person does not have diabetes and `1` means the person has diabetes.


In [ ]:
diabetes_data = diabetes_data.drop_duplicates().copy()

if "diabetes" not in diabetes_data.columns:
    raise KeyError("The dataset must contain a column named 'diabetes'.")

X = diabetes_data.drop(columns=["diabetes"])
y = diabetes_data["diabetes"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())


## 4. Create the Train/Test Split

The data is divided into training and testing sets using an 80/20 split. Stratification is used so that the percentage of diabetes cases remains similar in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training observations: {X_train.shape[0]}")
print(f"Testing observations: {X_test.shape[0]}")


## 5. Preprocess the Features

The dataset contains both numerical and categorical variables. Missing numerical values are filled with the median and then standardized. Missing categorical values are filled with the most common category and converted into numerical form using one-hot encoding.


In [ ]:
numerical_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X.select_dtypes(exclude=["number"]).columns.tolist()

numerical_preprocessing = Pipeline(
    steps=[
        ("missing_value_handler", SimpleImputer(strategy="median")),
        ("standardization", StandardScaler())
    ]
)

categorical_preprocessing = Pipeline(
    steps=[
        ("missing_value_handler", SimpleImputer(strategy="most_frequent")),
        ("one_hot_encoding", OneHotEncoder(handle_unknown="ignore", drop="first"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_preprocessing, numerical_columns),
        ("categorical", categorical_preprocessing, categorical_columns)
    ]
)

print(f"Numerical columns: {len(numerical_columns)}")
print(f"Categorical columns: {len(categorical_columns)}")


## 6. Train Four Classification Models

The following models are trained using the same training data and preprocessing steps:

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- K-Nearest Neighbors

Using the same split and preprocessing pipeline makes the comparison fair.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=7
    )
}

model_results = []
trained_models = {}
test_predictions = {}

for model_name, classifier in models.items():
    model_pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            ("classifier", classifier)
        ]
    )

    model_pipeline.fit(X_train, y_train)
    predicted_classes = model_pipeline.predict(X_test)

    if hasattr(model_pipeline.named_steps["classifier"], "predict_proba"):
        predicted_probabilities = model_pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, predicted_probabilities)
    else:
        roc_auc = np.nan

    accuracy = accuracy_score(y_test, predicted_classes)
    precision = precision_score(y_test, predicted_classes, zero_division=0)
    recall = recall_score(y_test, predicted_classes, zero_division=0)
    f1 = f1_score(y_test, predicted_classes, zero_division=0)

    model_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

    trained_models[model_name] = model_pipeline
    test_predictions[model_name] = predicted_classes

print("All four classification models were trained successfully.")


## 7. Metrics Comparison Table

The table below compares all four models. Higher values are better for Accuracy, Precision, Recall, F1-score, and ROC-AUC.


In [ ]:
comparison_table = pd.DataFrame(model_results)

comparison_table = comparison_table.sort_values(
    by=["F1-score", "Recall", "Accuracy"],
    ascending=False
).reset_index(drop=True)

comparison_table


## 8. Select the Best Model

The best model is selected using F1-score as the main measure because the dataset is imbalanced and diabetes cases are less common than non-diabetes cases. Recall and Accuracy are used as additional comparison measures.


In [ ]:
best_model_row = comparison_table.iloc[0]
best_model_name = best_model_row["Model"]

print(f"Selected best model: {best_model_name}")
print(best_model_row)


## 9. Confusion Matrix for the Best Model

The confusion matrix shows how many diabetes and non-diabetes cases were classified correctly and incorrectly by the selected model.


In [ ]:
best_predictions = test_predictions[best_model_name]
matrix = confusion_matrix(y_test, best_predictions)

display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["No Diabetes", "Diabetes"]
)

fig, ax = plt.subplots(figsize=(6, 5))
display.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()


## 10. Final Justification

In [ ]:
best_accuracy = best_model_row["Accuracy"]
best_precision = best_model_row["Precision"]
best_recall = best_model_row["Recall"]
best_f1 = best_model_row["F1-score"]
best_roc_auc = best_model_row["ROC-AUC"]

justification = (
    f"{best_model_name} was selected as the best model because it achieved the "
    f"strongest overall balance of performance, with an Accuracy of "
    f"{best_accuracy:.4f}, Precision of {best_precision:.4f}, Recall of "
    f"{best_recall:.4f}, and an F1-score of {best_f1:.4f}. "
    f"I gave more importance to the F1-score and Recall because correctly "
    f"identifying people with diabetes is more important than relying on "
    f"Accuracy alone in an imbalanced dataset."
)

if not np.isnan(best_roc_auc):
    justification += (
        f" Its ROC-AUC score of {best_roc_auc:.4f} also shows that the model "
        f"separates the two classes effectively."
    )

print(justification)


## Conclusion

This assignment compared four classification models using the same train/test split and preprocessing process. The final model was selected based on its F1-score, Recall, Accuracy, Precision, ROC-AUC, and confusion matrix performance.
